# Importar bibliotecas / Import Library

## Configuración del entorno de Smartsheet / Smartsheet Environment Setup

In [ ]:
#%pip install smartsheet-python-sdk

In [ ]:
import smartsheet
import pandas as pd
#from arcgis.gis import GIS
import arcpy
import datetime
import re
import os
from dotenv import load_dotenv, find_dotenv

from pathlib import Path
load_dotenv(find_dotenv())

In [ ]:
SMARTSHEET_ACCESS_TOKEN = os.environ.get('SMARTSHEET_TOKEN')
ss = smartsheet.Smartsheet(SMARTSHEET_ACCESS_TOKEN)
sheet_id_c1 = os.environ.get('SHEET_C1')
sheet_id_c2 = os.environ.get('SHEET_C2')
sheet_id_c3 = os.environ.get('SHEET_C3')

path_to_folder1 = os.environ.get('FOLDER_C1')
path_to_folder2 = os.environ.get('FOLDER_C2')
path_to_folder3 = os.environ.get('FOLDER_C3')


path_all = os.environ.get('SMARTSHEET_DATA_DIR')

default_gdb_path = os.environ.get('DEFAULT_GDB_PATH')

def smartsheet_to_dataframe(sheet):
    col_names = [col.title for col in sheet.columns]
    rows = []   
    for row in sheet.rows:
        cells = []
        for cell in row.cells:
            cells.append(cell.value)
        rows.append(cells)
    data_frame = pd.DataFrame(rows, columns=col_names)
    return data_frame

In [ ]:
# Add this right after your environment variable loading
print(f"default_gdb_path: {default_gdb_path}")
print(f"path_all: {path_all}")
print(f"path_to_folder1: {path_to_folder1}")
print(f"path_to_folder2: {path_to_folder2}")
print(f"path_to_folder3: {path_to_folder3}")

# Check if the path exists
if default_gdb_path:
    print(f"GDB path exists: {os.path.exists(default_gdb_path)}")
else:
    print("ERROR: default_gdb_path is None - check your .env file")

## Obtener datos de Smartsheet → Guardar como CSV → Subir CSV como tabla de ArcGIS / Get Smartsheet → Save as CSV → Upload CSV as ArcGIS table

**ES:** El nombre de la tabla seguirá el formato: `ss_C1_yyyymmddhhmm`

**EN:** The table name will follow the format: `ss_C1_yyyymmddhhmm`

### Componente 1 / Component 1

In [ ]:
sheet_c1 = ss.Sheets.get_sheet(sheet_id_c1)
df = smartsheet_to_dataframe(sheet_c1)
df = df[pd.notnull(df['CÓDIGO DE LA ACTIVIDAD'])]
df.head()

#save dataframe as a temporary csv file
ss_c1_csv = fr"{path_to_folder1}\ss_c1.csv"
df.to_csv(ss_c1_csv, encoding='utf-8-sig') #spanish encoding "utf-8-sig"
# Read the CSV file into a Pandas DataFrame
df = pd.read_csv(ss_c1_csv)

# Set the workspace
arcpy.env.workspace = fr"{default_gdb_path}"


timestamp = datetime.datetime.now()
time_string = timestamp.strftime("%Y-%m-%d %H:%M")

ss_name1 = "Ss_C1_tbl_" + str(time_string).replace("-","")
result1 = re.sub(r"[^\w_]+", "", ss_name1)
print(result1)

# Convert the CSV file to a table
arcpy.TableToTable_conversion(ss_c1_csv, arcpy.env.workspace, result1)

#save as date-time marker in same folder
df.to_csv(os.path.join(path_all,'{}.csv'.format(result1)), encoding='utf-8-sig')


### Componente 2 / Component 2

In [ ]:
### Componente 2
sheet_c2 = ss.Sheets.get_sheet(sheet_id_c2)
df2 = smartsheet_to_dataframe(sheet_c2)
df2 = df2[pd.notnull(df2['CÓDIGO DE LA ACTIVIDAD'])]
df2.head()

#save dataframe as a temporary csv file
ss_c2_csv = fr"{path_to_folder2}\ss_c2.csv"
df2.to_csv(ss_c2_csv, encoding='utf-8-sig') #spanish encoding "utf-8-sig"
# Read the CSV file into a Pandas DataFrame
df2 = pd.read_csv(ss_c2_csv)

# Set the workspace
arcpy.env.workspace = fr"{default_gdb_path}"

timestamp = datetime.datetime.now()
time_string = timestamp.strftime("%Y-%m-%d %H:%M")

ss_name2 = "Ss_C2_tbl_" + str(time_string).replace("-","")
result2 = re.sub(r"[^\w_]+", "", ss_name2)
print(result2)

# Convert the CSV file to a table
arcpy.TableToTable_conversion(ss_c2_csv, arcpy.env.workspace, result2)

#save as date-time marker in same folder
df2.to_csv(os.path.join(path_all,'{}.csv'.format(result2)), encoding='utf-8-sig')

### Componente 3 / Component 3

In [ ]:
### Componente 3
sheet_c3 = ss.Sheets.get_sheet(sheet_id_c3)
df3 = smartsheet_to_dataframe(sheet_c3)
df3 = df3[pd.notnull(df3['CÓDIGO DE LA ACTIVIDAD'])]
df3.head()

#save dataframe as a temporary csv file
ss_c3_csv = fr"{path_to_folder3}\ss_c3.csv"
df3.to_csv(ss_c3_csv, encoding='utf-8-sig') #spanish encoding "utf-8-sig"
# Read the CSV file into a Pandas DataFrame
df3 = pd.read_csv(ss_c3_csv)

# Set the workspace
arcpy.env.workspace = fr"{default_gdb_path}"

timestamp = datetime.datetime.now()
time_string = timestamp.strftime("%Y-%m-%d %H:%M")

ss_name3 = "Ss_C3_tbl_" + str(time_string).replace("-","")
result3 = re.sub(r"[^\w_]+", "", ss_name3)
print(result3)

# Convert the CSV file to a table
arcpy.TableToTable_conversion(ss_c3_csv, arcpy.env.workspace, result3)

#save as date-time marker in same folder
df3.to_csv(os.path.join(path_all,'{}.csv'.format(result3)), encoding='utf-8-sig')

### Fusionar tablas de C1 y C2 / Merge tables of C1 & C2

In [ ]:
### Merge tables of C1 & C2
# Define the input tables
input_tables = [result1, result2]

timestamp = datetime.datetime.now()
time_string = timestamp.strftime("%Y-%m-%d %H:%M")

# Define the output table name
name_time = "Ss_mrg_tbl_" + str(time_string).replace("-","")
output_table_name = re.sub(r"[^\w_]+", "", name_time)

print(output_table_name)
# Define the output table
output_table = os.path.join(default_gdb_path, output_table_name)

# Merge the tables
arcpy.Merge_management(input_tables, output_table)

# NO Ejecutar: comparar área de Smartsheet y área GIS usando groupby de Pandas / DO NOT Execute: compare Smartsheet area & GIS area using Pandas groupby

## Obtener tabla independiente de Smartsheet como DataFrame de Pandas / Get Smartsheet standalone table as Pandas DataFrame

In [ ]:
output_table

In [ ]:
import arcpy
import pandas as pd

# Specify the path to your standalone ArcGIS table that is from the Smartsheet
table_path = output_table

# Create a list of field names
field_names = [field.name for field in arcpy.ListFields(table_path)]

# Use a search cursor to extract the data from the table
data = []
with arcpy.da.SearchCursor(table_path, field_names) as cursor:
    for row in cursor:
        data.append(row)

# Create a Pandas DataFrame
df_ss_mgr_tbl = pd.DataFrame(data, columns=field_names)

In [ ]:
df_ss_mgr_area = df_ss_mgr_tbl.loc[:, ['CÓDIGO_DE_LA_ACTIVIDAD', 'TOTAL_DE_HECTÁREAS']]
df_ss_mgr_area.dropna()
df_ss_mgr_area = df_ss_mgr_area[df_ss_mgr_area['TOTAL_DE_HECTÁREAS'] != 0 ]
df_ss_mgr_area = df_ss_mgr_area.groupby(['CÓDIGO_DE_LA_ACTIVIDAD']).sum()
df_ss_mgr_area

## Obtener la tabla de la capa oficial de entidades como DataFrame de Pandas / Get Official Feature Layer table as Pandas DataFrame

In [ ]:
import geopandas as gpd

# Define the path to the GDB file and the layer name
gdb_path = r"C:\Users\yuny\OneDrive - IUCN International Union for Conservation of Nature\문서\ArcGIS\Packages\ShapesFinales_C1_54b09e\AR_Oficial_Acumulado.gdb"
official_point_layer_name = "AR_Oficial_punto_GTM"
official_polygon_layer_name = "AR_Oficial_poligono_GTM"

# Read the feature layer into a GeoDataFrame
gdf_point = gpd.read_file(gdb_path, layer=official_point_layer_name)
gdf_polygon = gpd.read_file(gdb_path, layer=official_polygon_layer_name)

# Convert the GeoDataFrame to a Pandas DataFrame
df_point = pd.DataFrame(gdf_point)
df_polygon = pd.DataFrame(gdf_polygon)

In [ ]:
gdf_point

In [ ]:
# Get only cdg, Area_ha from two layers
df_point = df_point.loc[:, ['CdgActvdd', 'Area_ha']]
df_point['Shape_Type'] = 'point'
df_point

In [ ]:
df_polygon = df_polygon.loc[:, ['CdgActvdd', 'Area_ha']]
df_polygon['Shape_Type'] = 'polygon'
df_polygon.shape[0] + df_point.shape[0]

In [ ]:
df_FL_mgr = pd.concat([df_point, df_polygon], ignore_index=True)

df_FL_mgr = df_FL_mgr.groupby(['CdgActvdd']).sum()

df_FL_mgr

## Comparar los dos conjuntos de datos / Compare the two datasets

In [ ]:
df_ss_mgr_area.compare()

In [ ]:
# Merge the dataframes based on the code columns
merged_df = pd.merge(df_ss_mgr_area, df_FL_mgr, left_on="CÓDIGO_DE_LA_ACTIVIDAD", right_on="CdgActvdd", how="outer" )

#Check if the areas match
#merged_df["Areas_Match"] = merged_df["TOTAL_DE_HECTÁREAS"] == merged_df["Area_ha"]
# Select the desired columns in the result
#result_df = merged_df[["CÓDIGO_DE_LA_ACTIVIDAD_x", "TOTAL_DE_HECTÁREAS", "CdgActvdd", "Area_ha", "Areas_Match"]]

# Rename the columns for clarity
#result_df.columns = ["CÓDIGO_DE_LA_ACTIVIDAD", "TOTAL_DE_HECTÁREAS", "CdgActvdd", "Area_ha", "Areas_Match"]

merged_df.columns


# Expresiones de cálculo de ArcGIS / ArcGIS Calculation expressions

In [ ]:
# list of 'TIPO DE TRATAMIENTO'



def trata(microcuenca):
    
    
    tratamiento = [
                    'Balanyá',
                    'Cotoyá',
                    'Eschaquichoj',
                    'Espumpujá',
                    'Joj',
                    'La Unión',
                    'Mactzul',
                    'Paguisis',
                    'Paxocol',
                    'Pinut',
                    'Pixcayá 1',
                    'Pixcayá-Papumay',
                    'Puxal',
                    'Quiejel',
                    'Sacabaj',
                    'SacGuexá',
                    'Sacputub A',
                    'Sepelá 1',
                    'Sepelá 2',
                    'Sigüilá',
                    'Tzunamá',
                    'Xayá-Coyolate',
                    'Xepocol',
                    'Xequijel'
                    ]

    control = [
                'Arriquib 196',
                'Chacap 157',
                'El Arco 176',
                'El Tumbadero 177',
                'Motagua Alto 156',
                'Pabacul 112',
                'Pachita 183',
                'Samalá 151',
                'Samalá 166',
                'Sepelá 154'
                ]


    if microcuenca in tratamiento:
        tipo_de_trata = "TRATAMIENTO"
    elif microcuenca in control:
        tipo_de_trata = "CONTROL"
    else:
        tipo_de_trata = ""
    return tipo_de_trata

In [ ]:
# Type of Area ID setting

def id_area(area):
    if area == "Influence":
        id_area = 3
    elif area == "Intervention":
        id_area = 2
    elif area == "Prioritized":
        id_area = 1
    else:
        id_area = 0
    return id_area

In [ ]:
#abe_ingles(!ACCIONES_DE_RESTAURACIÓN_AbE!)

def abe_eng(abe):
    if abe == "Sistema Agroforestal | Cultivos anuales":
        abe =  "1. AFS w annual Crops"
    elif abe == "Sistema Agroforestal | Cultivos perennes":
        abe =  "1. AFS w perennial Crops"
    elif abe == "Sistema Silvopastoril":
        abe = "Silvopastoril system"
    elif abe == "Plantaciones Forestales":
        abe = "2. Forest plantation"
    elif abe == "Bosque Natural con Fines de Producción":
        abe = "2. Natural forest production"
    elif abe == "Bosque Natural con Fines de Protección":
        abe = "3. Natural forest protection"
    elif abe == "Reforestación con Fines de Restauración":
        abe = "3. Natural forest restoration"
    else:
        abe = ""
    return abe

In [ ]:
# AbE to EbA (as english using calculate fiels (shift+alt+c))



def abe(abe_long):
    #AbE value abbreviation
    #column_ABE = ssheet.get_column_by_title("ACCIONES DE RESTAURACIÓN AbE")
    abe_abbr = abe_long #row.get_column(column_ABE.id_).value
    if abe_abbr == "Sistema Agroforestal | Cultivos anuales":
        abe_abbr =  "anual"
    elif abe_abbr == "Sistema Agroforestal | Cultivos perennes":
        abe_abbr =  "peren"
    elif abe_abbr == "Sistema Silvopastoril":
        abe_abbr = "silvo"
    elif abe_abbr == "Plantaciones Forestales":
        abe_abbr = "plant"
    elif abe_abbr == "Bosque Natural con Fines de Producción":
        abe_abbr = "produ"
    elif abe_abbr == "Bosque Natural con Fines de Protección":
        abe_abbr = "prote"
    elif abe_abbr == "Reforestación con Fines de Restauración":
        abe_abbr = "refor"
    else:
        abe_abbr = "error"
    return abe_abbr



In [ ]:

#PPD: small grants
#PMD: medium grants



def grants(contrato_org):
    if contrato_org is None:
        pass
    else:
        split_values = contrato_org.split('-')
        
        if split_values[1] == 'PPD':
            grants = "Small Grants"
        elif split_values[1] == 'PMD':
            grants = "Medium Grants"
        else:
            grants = ""
        return grants
    
    

In [ ]:
# ArcPy in ArcGIS pro


arcpy.management.CalculateField(
    in_table="all_new_merged_PairwiseInter1",
    field="Type_of_Grants",
    expression="grants(!NÚMERO_DE_CONTRATO!)",
    expression_type="PYTHON3",
    code_block="""def grants(contrato_org):
    if contrato_org is None:
        return ""
    else:
        split_values = contrato_org.split('-')
        
        if split_values[1] == 'PPD':
            grants = "Small Grants"
        elif split_values[1] == 'PMD':
            grants = "Medium Grants"
        else:
            grants = ""
        return grants
    """,
    field_type="TEXT",
    enforce_domains="NO_ENFORCE_DOMAINS"
)

In [ ]:
def cada_punto(total, parcelas):
    if parcelas != Null:
        return total / parcelas
    else :
        return Null

### Obtener capa de entidades o tabla de GDB como DataFrame de Pandas / Get GDB feature layer or table as Pandas DataFrame

In [ ]:
# Set the workspace
arcpy.env.workspace = r"C:\Users\yuny\OneDrive - IUCN International Union for Conservation of Nature\문서\ArcGIS\Packages\ShapesFinales_C1_54b09e\p30\default.gdb"

# Get a list of feature layers in the geodatabase
feature_layers = arcpy.ListFeatureClasses()
tables = arcpy.ListTables()
all_class = feature_layers + tables


In [ ]:
# Get list in gdb
feature_layers.sort()
feature_layers

all_class.sort()
all_class

In [ ]:
# Get dataFrame from a specific feature class or table

#name that you want to see data in gdb of ArcGIS pro
feature_class = "AR_Annual_Report__Export_Merge"

# Get the field names
fields = [field.name for field in arcpy.ListFields(feature_class)]

# Use the SearchCursor to extract the data
data = []
with arcpy.da.SearchCursor(feature_class, fields) as cursor:
    for row in cursor:
        data.append(row)

# Convert the data to a Pandas DataFrame        
df = pd.DataFrame(data, columns=fields)

In [ ]:
df['Area_ha'].describe()